In [1]:
import pandas as pd
import json

df = pd.read_excel("train_fixed.xlsx")
print(df.head())

   review_id                                        review_text  star_rating  \
0       7238                لا يوجد الدفع بالبطاقه عند الاستلام            3   
1       1036  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...            5   
2       1975  تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...            1   
3       3024                                    احلي مكان فزايد            5   
4       5483  الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...            4   

                  date              business_name      business_category  \
0  2026-03-08 00:00:00                       Noon              ecommerce   
1        قبل يومين (2)      ممشي مصر Mawlana Cafe                  كافيه   
2              قبل شهر          بيت لحم Beet Lahm                   مطعم   
3              قبل شهر  ذا بلكون كافيه الشيخ زايد  مطعم مأكولات ومشروبات   
4              قبل سنة        The Best Restaurant                   مطعم   

      platform                                 aspects  \
0   

In [2]:
print(df.columns)

Index(['review_id', 'review_text', 'star_rating', 'date', 'business_name',
       'business_category', 'platform', 'aspects', 'aspect_sentiments'],
      dtype='object')


In [3]:
# Convert JSON strings into real objects
df["aspects"] = df["aspects"].apply(json.loads)
df["aspect_sentiments"] = df["aspect_sentiments"].apply(json.loads)

In [4]:
# Convert ABSA into ML format (one row per aspect)
rows = []

for _, row in df.iterrows():
    review = row["review_text"]
    sentiments = row["aspect_sentiments"]

    for aspect, sentiment in sentiments.items():
        rows.append({
            "text": review,
            "aspect": aspect,
            "label": sentiment
        })

train_df = pd.DataFrame(rows)
print(train_df.head())
print(f"Total rows: {len(train_df)}")

                                                text          aspect     label
0                لا يوجد الدفع بالبطاقه عند الاستلام  app_experience  negative
1                لا يوجد الدفع بالبطاقه عند الاستلام        delivery  negative
2  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...     cleanliness  positive
3  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...        ambiance  positive
4  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...         service  positive
Total rows: 3333


In [5]:
# Convert labels to numbers
label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

train_df["label_id"] = train_df["label"].map(label_map)
print(train_df[["label", "label_id"]].value_counts())

label     label_id
positive  2           1646
negative  0           1538
neutral   1            149
Name: count, dtype: int64


In [6]:
# Combine text + aspect into one input string
train_df["input"] = train_df["text"] + " [SEP] " + train_df["aspect"]
print(train_df["input"].iloc[0])

لا يوجد الدفع بالبطاقه عند الاستلام [SEP] app_experience


In [7]:
# Load Arabic BERT model and tokenizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "aubmindlab/bert-base-arabertv2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)
print("Model loaded successfully")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully


In [8]:
# Tokenization
encodings = tokenizer(
    list(train_df["input"]),
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)
print("input_ids shape:", encodings["input_ids"].shape)

input_ids shape: torch.Size([3333, 128])


In [10]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42
)

print(f"train: {len(train_data)}")
print(f"val:   {len(val_data)}")

train: 2666
val:   667
